# Detecting inherent linearity in large CNNs

This Notebook serves to experiment with detecting inherent linearity in ResNets. These experiments will be run on smaller instances of ResNets to test functionality and provide evidence for a feasibility study as part of the graduation preparation phase. Larger experiments for the final paper(s) will be handled using Python files in this repository.

In [1]:
import torch
from torchvision.models import resnet18
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from tqdm import tqdm

print(torch.cuda.is_available())

True


In [2]:
model = resnet18()
test_dataset = CIFAR10(root="./data", train=False, download=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=True, num_workers=2)

"**Mean of preactivations $\bar{p}^l$** We denote the distribution of inputs $z$ to a nonlinear activation function $f(z)$ as the preactivations. For each activation function/node, we compute the mean of the preactivations, and then we compute another mean of these values per layer $l$: $\bar{p}^l= \frac{1}{M}\sum_{i=1}^M \left(\frac{1}{N}\sum_{s=1}^N z_{s,i}^l\right)$,
with $M$ the number of nodes in layer $l$ and $N$ the number of samples, and $z_{s,i}^l$ the preactivation value for sample $s$ at node $i$ at layer $l$. We compute this value through randomly selecting 500 samples of the input data instead of the whole dataset, which significantly reduces the computational cost." (Pinson et al., 2024, p. 3).

In case there is BatchNormalization applied between the convolution and the activation function, the mean of the preactivations will be approximately 0, due to the normalization. However, BN has two learned parameters per channel, namely a scaling and a shifting parameter. The shifting parameter can be used to recover the mean of the preactivations before BN. Therefore, in case of BN, we will use the shifting parameter as the mean of the preactivations. (Pinson et al., 2024)

In [ ]:
def mean_preactivations(model, data_loader, device='cuda'):
    """Compute the mean of preactivations for each layer in the model. Returns a dictionary with layer names as keys and mean preactivation values as values.
    Args:
        model: The neural network model.
        data_loader: DataLoader for the input data.
        device: Device to run the computations on.

    Returns:
        dict: A dictionary with layer names as keys and mean preactivation values as values.
    """
    model.to(device)
    model.eval()

    mean_preactivations = {}
    hooks = []

    def get_hook(name):
        def hook(module, input, output):
            if isinstance(module, torch.nn.ReLU):
                preactivations = input[0].detach().cpu()
                mean_preactivation = preactivations.mean().item()
                mean_preactivations[name] = mean_preactivation
            elif isinstance(module, torch.nn.BatchNorm2d):
                mean_shift = module.bias.detach().cpu().mean().item()
                mean_preactivations[name] = mean_shift
        return hook

    for name, module in tqdm(model.named_modules(), desc="Registering hooks", leave=False):
        if isinstance(module, (torch.nn.ReLU, torch.nn.BatchNorm2d)):
            hooks.append(module.register_forward_hook(get_hook(name)))
    with torch.no_grad():
        for inputs, _ in tqdm(data_loader, desc="Computing mean preactivations", leave=False):
            inputs = inputs.to(device)
            model(inputs)
            break  # Only need one batch for mean preactivations
    for hook in tqdm(hooks, desc="Removing hooks", leave=False):
        hook.remove()
    return mean_preactivations


mean_preacts = mean_preactivations(model, test_loader)
print(mean_preacts)

Computing mean preactivations:   0%|          | 0/1250 [00:00<?, ?it/s]